# ViFinQA - Schema linking v3 consensus 2/3

Kaggle: GPU T4 x2, Internet On. Attach only `kaggle-payload-schema-linking-v3-w010`.

In [ ]:
import json, pathlib

PAYLOAD = "/kaggle/input/datasets/kien2005/kaggle-payload-schema-linking-v3-w010"
payload = pathlib.Path(PAYLOAD)
retrieval_path = payload / "retrieval.jsonl"
manifest_path = payload / "payload-manifest.json"
assert retrieval_path.exists(), f"Missing retrieval: {retrieval_path}"
assert manifest_path.exists(), f"Missing payload manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest.get("schema_version") == 2
assert len(manifest.get("files", {})) == 251
assert sum(1 for _ in retrieval_path.open(encoding="utf-8")) == 1012
print("PAYLOAD =", PAYLOAD, "| verified files =", len(manifest["files"]))

import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator first"
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
import pathlib, shutil

SRC = pathlib.Path(PAYLOAD) / "code"
DST = pathlib.Path("/kaggle/working/code")
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print("code ->", DST)

In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes
print("transformers", transformers.__version__, "bitsandbytes", bitsandbytes.__version__)

In [ ]:
%%time
# Smoke test only; majority voting must reject split selections.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target empty \
    --out /kaggle/working/codegen_schema_linking_v3_consensus_smoke.jsonl --limit 32 \
    --n 3 --temperature 0.35 --k 15 --max-tokens 96 --batch-size 4 \
    --checkpoint-every 8 --time-budget-min 30 --seed 13 --no-resume

In [ ]:
import collections, json
rows = [json.loads(line) for line in open("/kaggle/working/codegen_schema_linking_v3_consensus_smoke.jsonl", encoding="utf-8")]
assert len(rows) == 32
selected = [row for row in rows if row["source"] == "llm_select"]
assert all(row.get("votes", 0) >= 2 for row in selected)
assert all("consensus=" in row.get("detail", "") for row in selected)
print(collections.Counter(row["source"] for row in rows))
print("consensus selections", len(selected))

In [ ]:
%%time
# Only deterministic-empty questions enter the LLM queue.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target empty \
    --out /kaggle/working/codegen_schema_linking_v3_consensus_empty_sel7b_k15_n3.jsonl \
    --n 3 --temperature 0.35 --k 15 --max-tokens 96 --batch-size 4 \
    --checkpoint-every 16 --time-budget-min 400 --seed 13

In [ ]:
import collections, json, math, pathlib

out = pathlib.Path("/kaggle/working/codegen_schema_linking_v3_consensus_empty_sel7b_k15_n3.jsonl")
rows = [json.loads(line) for line in out.open(encoding="utf-8")]
ids = [row["id"] for row in rows]
assert len(rows) == 1012 and len(set(ids)) == 1012
assert set(ids) == set(range(1, 1013))
assert all(math.isfinite(float(row["answer"])) for row in rows)
signatures = {row.get("run_signature", "") for row in rows if row.get("run_signature")}
assert len(signatures) == 1, signatures
selected = [row for row in rows if row["source"] == "llm_select"]
assert all(row.get("votes", 0) >= 2 for row in selected)
assert all("consensus=" in row.get("detail", "") for row in selected)
print(collections.Counter(row["source"] for row in rows))
print(collections.Counter(row.get("votes", 0) for row in selected))
print("run signature", next(iter(signatures))[:16])
print("OK: 1012 rows; consensus candidates =", len(selected), "->", out)

Download `/kaggle/working/codegen_schema_linking_v3_consensus_empty_sel7b_k15_n3.jsonl` after the final check passes.